In [1]:
import pandas as pd
import h2o
from h2o.automl import H2OAutoML

In [3]:
data = pd.read_csv('moneyball.csv')
data.head()

,team,league,year,rs,ra,w,obp,slg,ba,playoffs,rankseason,rankplayoffs,g,oobp,oslg
0,ARI,NL,2012,734,688,81,0.328,0.418,0.259,0,NaN,NaN,162,0.317,0.415
1,ATL,NL,2012,700,600,94,0.320,0.389,0.247,1,4.0,5.0,162,0.306,0.378
2,BAL,AL,2012,712,705,93,0.311,0.417,0.247,1,5.0,4.0,162,0.315,0.403
3,BOS,AL,2012,734,806,69,0.315,0.415,0.260,0,NaN,NaN,162,0.331,0.428
4,CHC,NL,2012,613,759,61,0.302,0.378,0.240,0,NaN,NaN,162,0.335,0.424


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1232 entries, 0 to 1231
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   team          1232 non-null   object 
 1   league        1232 non-null   object 
 2   year          1232 non-null   int64  
 3   rs            1232 non-null   int64  
 4   ra            1232 non-null   int64  
 5   w             1232 non-null   int64  
 6   obp           1232 non-null   float64
 7   slg           1232 non-null   float64
 8   ba            1232 non-null   float64
 9   playoffs      1232 non-null   int64  
 10  rankseason    244 non-null    float64
 11  rankplayoffs  244 non-null    float64
 12  g             1232 non-null   int64  
 13  oobp          420 non-null    float64
 14  oslg          420 non-null    float64
dtypes: float64(7), int64(6), object(2)
memory usage: 144.5+ KB


In [5]:
data.isna().sum()

team              0
league            0
year              0
rs                0
ra                0
w                 0
obp               0
slg               0
ba                0
playoffs          0
rankseason      988
rankplayoffs    988
g                 0
oobp            812
oslg            812
dtype: int64

## H2O

In [2]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.501-b08, mixed mode)
  Starting server from C:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\TARLAN~1.CAB\AppData\Local\Temp\tmpl148jt_2
  JVM stdout: C:\Users\TARLAN~1.CAB\AppData\Local\Temp\tmpl148jt_2\h2o_tarlan_cabiyev_started_from_python.out
  JVM stderr: C:\Users\TARLAN~1.CAB\AppData\Local\Temp\tmpl148jt_2\h2o_tarlan_cabiyev_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Asia/Baku
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 15 days
H2O_cluster_name:,H2O_from_python_tarlan_cabiyev_tl75of
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.024 Gb
H2O_cluster_total_cores:,0
H2O_cluster_allowed_cores:,0
H2O_cluster_status:,"locked, healthy"


In [6]:
h2o_data = h2o.H2OFrame(data)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [7]:
type(h2o_data)

h2o.frame.H2OFrame

## Datanın bölünməsi

In [8]:
train, valid, test = h2o_data.split_frame(ratios=[0.8, 0.1], seed=123)

In [19]:
target = 'rs'
features = data.drop(columns=[target]).columns.to_list()

## AutoML

In [27]:
model = H2OAutoML(
    stopping_metric = "mae",
    nfolds = 5,
    seed = 123,
    max_runtime_secs = 180
)

model.train(
    x = features,
    y = target,
    training_frame = train,
    validation_frame = valid,
    leaderboard_frame = test
)

AutoML progress: |


18:39:55.540: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
18:39:55.542: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2ODeepLearningEstimator : Deep Learning
Model Key: DeepLearning_grid_1_AutoML_2_20260806_183955_model_9


Status of Neuron Layers: predicting rs, regression, gaussian distribution, Quadratic loss, 25,801 weights/biases, 311.2 KB, 8,598 training samples, mini-batch size 1
    layer    units    type              dropout    l1    l2    mean_rate    rate_rms    momentum    mean_weight    weight_rms    mean_bias    bias_rms
--  -------  -------  ----------------  ---------  ----  ----  -----------  ----------  ----------  -------------  ------------  -----------  ------------
    1        256      Input             10
    2        100      RectifierDropout  30         0     0     0.115953     0.265637    0           -0.00137518    0.0754381     0.262626     0.140566
    3        1        Linear                       0     0     0.00348415   0.00137103  0           -0.000700454   0.0930535     0.0155958    1.09713e-154

ModelMetricsRegression: deeplearning
** Reported on train data. **

MSE: 451.7186393064462
RMSE: 21.253673548505592
MAE: 16.85140931786907
RMSLE: 0.02993795891267437
Mean Residual Deviance: 451.7186393064462

ModelMetricsRegression: deeplearning
** Reported on validation data. **

MSE: 457.446176607715
RMSE: 21.387991411250262
MAE: 17.437595186087467
RMSLE: 0.030797028577266114
Mean Residual Deviance: 457.446176607715

ModelMetricsRegression: deeplearning
** Reported on cross-validation data. **

MSE: 690.3160344559897
RMSE: 26.2738659975267
MAE: 20.879575895860835
RMSLE: 0.03694936441720748
Mean Residual Deviance: 690.3160344559897

Cross-Validation Metrics Summary: 
                        mean       sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  ---------  ----------  ------------  ------------  ------------  ------------  ------------
aic                     nan        0           nan           nan           nan           nan           nan
loglikelihood           nan        0           nan           nan           nan           nan           nan
mae                     20.8816    1.12493     20.119        19.4982       21.0284       21.3588       22.4038
mean_residual_deviance  690.437    68.4348     655.008       621.688       676.647       696.488       802.352
mse                     690.437    68.4348     655.008       621.688       676.647       696.488       802.352
r2                      0.916582   0.00819273  0.925056      0.91759       0.909689      0.923797      0.906779
residual_deviance       690.437    68.4348     655.008       621.688       676.647       696.488       802.352
rmse                    26.2512    1.2796      25.5931       24.9337       26.0124       26.3911       28.3258
rmsle                   0.0369059  0.00209164  0.0362809     0.0342179     0.0363983     0.0377376     0.0398949

Scoring History: 
    timestamp            duration          training_speed    epochs    iterations    samples    training_rmse    training_deviance    training_mae    training_r2    validation_rmse    validation_deviance    validation_mae    validation_r2
--  -------------------  ----------------  ----------------  --------  ------------  ---------  ---------------  -------------------  --------------  -------------  -----------------  ---------------------  ----------------  ---------------
    2026-08-06 18:41:59  0.000 sec                           0         0             0          nan              nan                  nan             nan            nan                nan                    nan               nan
    2026-08-06 18:41:59  1 min 52.048 sec  127333 obs/sec    0.781186  1             764        29.485           869.368              23.7651         0.895413       29.321             859.723                24.3455           0.908703
    2026-08-06 18:41:59  1 min 52.094 sec  186913 obs/sec    8.79141   11            8598       21.2537          451.719              16.8514         0.945657       21.388         

In [28]:
model.leaderboard.as_data_frame()

c:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,DeepLearning_grid_1_AutoML_2_20260806_183955_m...,23.245421,540.349575,18.984114,0.032426,540.349575
1,StackedEnsemble_AllModels_3_AutoML_2_20260806_...,23.567299,555.417585,19.307594,0.033243,555.417585
2,DeepLearning_1_AutoML_2_20260806_183955,23.632950,558.516341,19.374945,0.033286,558.516341
3,DeepLearning_grid_1_AutoML_2_20260806_183955_m...,24.066351,579.189251,19.238756,0.033296,579.189251
4,StackedEnsemble_BestOfFamily_3_AutoML_2_202608...,24.754578,612.789111,20.340217,0.034756,612.789111
...,...,...,...,...,...,...
69,DeepLearning_grid_3_AutoML_2_20260806_183955_m...,34.139808,1165.526488,27.286396,0.048371,1165.526488
70,DeepLearning_grid_2_AutoML_2_20260806_183955_m...,35.498360,1260.133534,29.096250,0.051751,1260.133534
71,DeepLearning_grid_2_AutoML_2_20260806_183955_m...,41.492650,1721.640014,32.499765,0.061582,1721.640014
72,DeepLearning_grid_3_AutoML_2_20260806_183955_m...,55.619952,3093.579096,43.562071,0.081740,3093.579096


In [29]:
best_model = model.leader
best_model

Model Details
=============
H2ODeepLearningEstimator : Deep Learning
Model Key: DeepLearning_grid_1_AutoML_2_20260806_183955_model_9


Status of Neuron Layers: predicting rs, regression, gaussian distribution, Quadratic loss, 25,801 weights/biases, 311.2 KB, 8,598 training samples, mini-batch size 1
    layer    units    type              dropout    l1    l2    mean_rate    rate_rms    momentum    mean_weight    weight_rms    mean_bias    bias_rms
--  -------  -------  ----------------  ---------  ----  ----  -----------  ----------  ----------  -------------  ------------  -----------  ------------
    1        256      Input             10
    2        100      RectifierDropout  30         0     0     0.115953     0.265637    0           -0.00137518    0.0754381     0.262626     0.140566
    3        1        Linear                       0     0     0.00348415   0.00137103  0           -0.000700454   0.0930535     0.0155958    1.09713e-154

ModelMetricsRegression: deeplearning
** Reported on train data. **

MSE: 451.7186393064462
RMSE: 21.253673548505592
MAE: 16.85140931786907
RMSLE: 0.02993795891267437
Mean Residual Deviance: 451.7186393064462

ModelMetricsRegression: deeplearning
** Reported on validation data. **

MSE: 457.446176607715
RMSE: 21.387991411250262
MAE: 17.437595186087467
RMSLE: 0.030797028577266114
Mean Residual Deviance: 457.446176607715

ModelMetricsRegression: deeplearning
** Reported on cross-validation data. **

MSE: 690.3160344559897
RMSE: 26.2738659975267
MAE: 20.879575895860835
RMSLE: 0.03694936441720748
Mean Residual Deviance: 690.3160344559897

Cross-Validation Metrics Summary: 
                        mean       sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  ---------  ----------  ------------  ------------  ------------  ------------  ------------
aic                     nan        0           nan           nan           nan           nan           nan
loglikelihood           nan        0           nan           nan           nan           nan           nan
mae                     20.8816    1.12493     20.119        19.4982       21.0284       21.3588       22.4038
mean_residual_deviance  690.437    68.4348     655.008       621.688       676.647       696.488       802.352
mse                     690.437    68.4348     655.008       621.688       676.647       696.488       802.352
r2                      0.916582   0.00819273  0.925056      0.91759       0.909689      0.923797      0.906779
residual_deviance       690.437    68.4348     655.008       621.688       676.647       696.488       802.352
rmse                    26.2512    1.2796      25.5931       24.9337       26.0124       26.3911       28.3258
rmsle                   0.0369059  0.00209164  0.0362809     0.0342179     0.0363983     0.0377376     0.0398949

Scoring History: 
    timestamp            duration          training_speed    epochs    iterations    samples    training_rmse    training_deviance    training_mae    training_r2    validation_rmse    validation_deviance    validation_mae    validation_r2
--  -------------------  ----------------  ----------------  --------  ------------  ---------  ---------------  -------------------  --------------  -------------  -----------------  ---------------------  ----------------  ---------------
    2026-08-06 18:41:59  0.000 sec                           0         0             0          nan              nan                  nan             nan            nan                nan                    nan               nan
    2026-08-06 18:41:59  1 min 52.048 sec  127333 obs/sec    0.781186  1             764        29.485           869.368              23.7651         0.895413       29.321             859.723                24.3455           0.908703
    2026-08-06 18:41:59  1 min 52.094 sec  186913 obs/sec    8.79141   11            8598       21.2537          451.719              16.8514         0.945657       21.388         

## Reqressiya Modelinin Perormansının Qiymətləndirilməsi

In [30]:
best_model.model_performance(train=True).r2()

0.9456571328508591

In [31]:
best_model.model_performance(valid=True).r2()

0.9514222054007915

In [32]:
best_model.model_performance(xval=True).r2()

0.9169532773565399

In [33]:
best_model.model_performance(test).r2()

0.928433283498883

## Proqnozlar

In [34]:
y_pred = best_model.predict(test).as_data_frame()
y_pred

deeplearning prediction progress: |██████████████████████████████████████████████| (done) 100%


c:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,predict
0,774.705227
1,756.920198
2,623.700507
3,678.764609
4,804.166616
...,...
109,664.614597
110,576.146732
111,465.166390
112,588.869211


In [35]:
pred = y_pred.predict
pred

0      774.705227
1      756.920198
2      623.700507
3      678.764609
4      804.166616
          ...    
109    664.614597
110    576.146732
111    465.166390
112    588.869211
113    701.936583
Name: predict, Length: 114, dtype: float64